# Setup

In [ ]:
from pathlib import Path
import pandas as pd

import keypoint_moseq as kpms

WORKDIR = Path.cwd().parent
dlc_project_dir = WORKDIR.parent / "dlc-pose-estimation" / "ElevatedMazeFood-Atanu-2026-04-04"
project_dir = WORKDIR / "results" / "ElevatedMazeFood"
config = kpms.load_config(str(project_dir))
model_name = project_dir / "multi_fit_20260708-13"

# load data (e.g. from DeepLabCut)
keypoint_data_path = str(dlc_project_dir / "raw_pose_data")  # can be a file, a directory, or a list of files
coordinates, confidences, bodyparts = kpms.load_keypoints(keypoint_data_path, "deeplabcut")

In [ ]:
FPS = 15.0
MIN_FREQUENCY = 0.005

# Assign Groups

In [ ]:
import inspect, keypoint_moseq as kpms
print(inspect.getsource(kpms.interactive_group_setting))

In [ ]:
kpms.interactive_group_setting(str(project_dir), str(model_name))

# Generate dataframes

In [ ]:
moseq_df = kpms.compute_moseq_df(project_dir, model_name, smooth_heading=True)
moseq_df

In [ ]:
stats_df = kpms.compute_stats_df(
    project_dir,
    model_name,
    moseq_df,
    min_frequency=MIN_FREQUENCY,  # threshold frequency for including a syllable in the dataframe
    groupby=["group", "name"],  # column(s) to group the dataframe by
    fps=FPS,
)  # frame rate of the video from which keypoints were inferred

stats_df

# Label syllables

In [ ]:
kpms.label_syllables(str(project_dir), str(model_name), moseq_df)

# Compare between groups

In [ ]:
kpms.plot_syll_stats_with_sem(
    stats_df,
    project_dir,
    model_name,
    plot_sig=True,  # whether to mark statistical significance with a star
    thresh=0.05,  # significance threshold
    stat="frequency",  # statistic to be plotted (e.g. 'duration' or 'velocity_px_s_mean')
    order="stat",  # order syllables by overall frequency ("stat") or degree of difference ("diff")
    ctrl_group="a",  # name of the control group for statistical testing
    exp_group="b",  # name of the experimental group for statistical testing
    figsize=(8, 4),  # figure size
    groups=stats_df["group"].unique(),  # groups to be plotted
);

## Transition matrices

In [ ]:
normalize = "bigram"  # normalization method ("bigram", "rows" or "columns")

trans_mats, usages, groups, syll_include = kpms.generate_transition_matrices(
    project_dir,
    model_name,
    normalize=normalize,
    min_frequency=MIN_FREQUENCY,  # minimum syllable frequency to include
)

kpms.visualize_transition_bigram(
    project_dir,
    model_name,
    groups,
    trans_mats,
    syll_include,
    normalize=normalize,
    show_syllable_names=True,  # label syllables by index (False) or index and name (True)
)

## Syllable Transition Graph

In [ ]:
kpms.plot_transition_graph_group(
    project_dir,
    model_name,
    groups,
    trans_mats,
    usages,
    syll_include,
    layout="circular",  # transition graph layout ("circular" or "spring")
    show_syllable_names=False,  # label syllables by index (False) or index and name (True)
)

In [ ]:
# Generate a difference-graph for each pair of groups.

kpms.plot_transition_graph_difference(
    project_dir, model_name, groups, trans_mats, usages, syll_include, layout="circular"
) 